# PhoBERT Baseline with Positional Encoding (Fair Comparison - Kaggle)

## Uniform Config for Fair Comparison
- **Model**: PhoBERT only (no auxiliary features)
- **Pooling**: Mean Pooling + Positional Encoding
- **Batch Size**: 16
- **Class Weights**: sqrt_balanced
- **Gradual Unfreezing**: Yes
- **Dropout**: 0.3
- **Head LR**: 5e-5
- **Selection Metric**: F1 Macro
- **Early Stop**: 2
- **Epochs**: 10

## 1. Setup and Imports

In [13]:
import os
import sys
import json
import time
import random
import copy
import re

from pathlib import Path
from datetime import datetime
from collections import Counter


def find_kaggle_project_root():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        raise FileNotFoundError('/kaggle/input not found')

    # Preferred structure: /kaggle/input/<dataset>/data/processed
    for ds in input_root.iterdir():
        if ds.is_dir() and (ds / 'data' / 'processed').exists():
            return ds

    # Fallback: deep scan for data/processed
    for processed_dir in input_root.rglob('processed'):
        if processed_dir.is_dir() and processed_dir.parent.name == 'data':
            return processed_dir.parent.parent

    raise FileNotFoundError(
        'Cannot find Kaggle dataset root containing data/processed. '
        f'Available folders: {list(input_root.glob("*"))}'
    )


PROJECT_ROOT = find_kaggle_project_root()
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Kaggle inputs = {list(Path("/kaggle/input").glob("*"))}')

PROJECT_ROOT = /kaggle/input/datasets/jadt145/phobert
Kaggle inputs = [PosixPath('/kaggle/input/datasets')]


In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    precision_score, recall_score, precision_recall_fscore_support,
)
from tqdm import tqdm
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')


# ============================================
# INLINE DATA UTILITIES (no external dependency)
# ============================================

TEENCODE_DICT = {
    'ko': 'không', 'k': 'không', 'khong': 'không', 'hok': 'không', 'hk': 'không',
    'kg': 'không', 'k0': 'không', 'kô': 'không', 'hông': 'không',
    'j': 'gì', 'gj': 'gì', 'gi': 'gì',
    'm': 'mày', 'mik': 'mình', 'mk': 'mình', 'mìh': 'mình',
    't': 'tao', 'tui': 'tôi',
    'r': 'rồi', 'rùi': 'rồi', 'roi': 'rồi', 'ròi': 'rồi',
    'cx': 'cũng', 'cũng': 'cũng', 'cug': 'cũng', 'cũg': 'cũng',
    'đc': 'được', 'dc': 'được', 'đươc': 'được', 'duoc': 'được',
    'lm': 'lắm', 'lém': 'lắm', 'lem': 'lắm',
    'qá': 'quá', 'qa': 'quá', 'quá': 'quá', 'wa': 'quá',
    'bik': 'biết', 'bt': 'biết', 'bjết': 'biết',
    'thấy': 'thấy', 'thay': 'thầy', 'thâ': 'thầy',
    'e': 'em', 'a': 'anh',
    'v': 'vậy', 'vậy': 'vậy', 'vay': 'vậy',
    'vs': 'với', 'với': 'với',
    'ok': 'ok', 'oke': 'ok', 'okay': 'ok',
    'um': 'ừ', 'ừ': 'ừ', 'uh': 'ừ', 'uk': 'ừ',
    'bnh': 'bình', 'bth': 'bình thường', 'bthg': 'bình thường',
}


def normalize_teencode(text):
    """Normalize Vietnamese internet slang."""
    words = text.split()
    normalized_words = [TEENCODE_DICT.get(word, word) for word in words]
    return ' '.join(normalized_words)


def preprocess_vietnamese(text, normalize_slang=True):
    """Preprocess Vietnamese text."""
    text = text.lower()
    if normalize_slang:
        text = normalize_teencode(text)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)
    return ' '.join(text.split())


def load_data(data_dir, split):
    """Load UIT-VSFC dataset from processed directory."""
    split_dir = os.path.join(data_dir, split)
    sents_path = os.path.join(split_dir, 'sents.txt')
    labels_path = os.path.join(split_dir, 'sentiments.txt')
    
    with open(sents_path, 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f.readlines()]
    with open(labels_path, 'r', encoding='utf-8') as f:
        labels = [int(line.strip()) for line in f.readlines()]
    
    return texts, labels


# ============================================
# SEED & DEVICE SETUP
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Configuration (Uniform for Fair Comparison)

In [15]:
class Config:
    BASE_DIR = str(PROJECT_ROOT)
    DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')

    MODEL_TYPE = 'PhoBERT_Baseline_Positional_Fair'
    EXPERIMENT_TYPE = 'improvements'
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

    RESULTS_ROOT = '/kaggle/working'
    RESULTS_DIR = os.path.join(RESULTS_ROOT, 'results', MODEL_TYPE, EXPERIMENT_TYPE, TIMESTAMP)
    MODELS_DIR = os.path.join(RESULTS_DIR, 'models')
    SUMMARIES_DIR = os.path.join(RESULTS_DIR, 'summaries')
    VISUALIZATIONS_DIR = os.path.join(RESULTS_DIR, 'visualizations')

    MODEL_NAME = 'vinai/phobert-base'
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    NUM_CLASSES = 3
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ===== UNIFORM CONFIG FOR FAIR COMPARISON =====
    MAX_LENGTH = 256
    BATCH_SIZE = 16
    EPOCHS = 10
    EARLY_STOP_PATIENCE = 2
    WARMUP_RATIO = 0.1
    GRADIENT_CLIP = 1.0
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.3

    # Learning Rates
    PHOBERT_LR_FROZEN = 0.0
    PHOBERT_LR_PARTIAL = 1e-5
    PHOBERT_LR_FULL = 2e-5
    HEAD_LR = 5e-5

    # Gradual Unfreezing
    FREEZE_EPOCHS = 1
    PARTIAL_UNFREEZE_EPOCH = 2
    FULL_UNFREEZE_EPOCH = 4
    PARTIAL_UNFREEZE_LAST_N = 4

    # Class Weights
    CLASS_WEIGHT_MODE = 'sqrt_balanced'

    # Selection Metric
    SELECTION_METRIC = 'f1_macro'

    # Classifier
    CLASSIFIER_HIDDEN_DIM = 256


config = Config()
for path in [config.RESULTS_DIR, config.MODELS_DIR, config.SUMMARIES_DIR, config.VISUALIZATIONS_DIR]:
    os.makedirs(path, exist_ok=True)

print('=' * 60)
print('BASELINE WITH POSITIONAL ENCODING - UNIFORM CONFIG')
print('=' * 60)
print(f'BASE_DIR: {config.BASE_DIR}')
print(f'DATA_DIR: {config.DATA_DIR}')
print(f'Results Dir: {config.RESULTS_DIR}')
print(f'Model: {config.MODEL_NAME}')
print(f'Batch Size: {config.BATCH_SIZE}')
print(f'Epochs: {config.EPOCHS}')
print(f'Head LR: {config.HEAD_LR}')
print(f'Dropout: {config.DROPOUT}')
print(f'Class Weights: {config.CLASS_WEIGHT_MODE}')
print(f'Device: {config.DEVICE}')

assert os.path.isdir(config.DATA_DIR), f'DATA_DIR not found: {config.DATA_DIR}'

BASELINE WITH POSITIONAL ENCODING - UNIFORM CONFIG
BASE_DIR: /kaggle/input/datasets/jadt145/phobert
DATA_DIR: /kaggle/input/datasets/jadt145/phobert/data/processed
Results Dir: /kaggle/working/results/PhoBERT_Baseline_Positional_Fair/improvements/20260321_151351
Model: vinai/phobert-base
Batch Size: 16
Epochs: 10
Head LR: 5e-05
Dropout: 0.3
Class Weights: sqrt_balanced
Device: cuda


## 3. Load Data (with Preprocessing)

In [16]:
# Load data
train_texts_raw, train_labels = load_data(config.DATA_DIR, 'train')
val_texts_raw, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts_raw, test_labels = load_data(config.DATA_DIR, 'test')

# Apply preprocessing (uniform for all models)
train_texts = [preprocess_vietnamese(text) for text in train_texts_raw]
val_texts = [preprocess_vietnamese(text) for text in val_texts_raw]
test_texts = [preprocess_vietnamese(text) for text in test_texts_raw]

print('Label distribution:')
for split_name, labels in [('Train', train_labels), ('Validation', val_labels), ('Test', test_labels)]:
    counter = Counter(labels)
    print(f'{split_name}: total={len(labels)}')
    for idx, name in config.LABEL_MAP.items():
        count = counter.get(idx, 0)
        print(f'  {name}: {count} ({count/len(labels)*100:.1f}%)')

Label distribution:
Train: total=11426
  Negative: 5325 (46.6%)
  Neutral: 458 (4.0%)
  Positive: 5643 (49.4%)
Validation: total=1583
  Negative: 705 (44.5%)
  Neutral: 73 (4.6%)
  Positive: 805 (50.9%)
Test: total=3166
  Negative: 1409 (44.5%)
  Neutral: 167 (5.3%)
  Positive: 1590 (50.2%)


## 4. Dataset & Model Definition

In [17]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_len = x.size(1)
        return self.dropout(x + self.pe[:, :seq_len, :].to(dtype=x.dtype))


class PhoBERTWithPositionalEncoding(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3, hidden_dim=256):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        phobert_dim = self.phobert.config.hidden_size

        self.positional_encoding = PositionalEncoding(
            d_model=phobert_dim,
            max_len=max(self.phobert.config.max_position_embeddings, 256),
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(phobert_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def get_head_modules(self):
        return [self.classifier]

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = self.positional_encoding(outputs.last_hidden_state)
        mask = attention_mask.unsqueeze(-1).type_as(token_embeddings)
        pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model = PhoBERTWithPositionalEncoding(
    model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES,
    dropout=config.DROPOUT, hidden_dim=config.CLASSIFIER_HIDDEN_DIM
).to(config.DEVICE)

print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Initial trainable parameters: {count_trainable_params(model):,}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters: 135,195,907
Initial trainable parameters: 135,195,907


## 5. Create DataLoaders

In [19]:
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, config.MAX_LENGTH)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, config.MAX_LENGTH)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, config.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 715
Val batches: 99
Test batches: 198


## 6. Training Setup

In [20]:
def build_class_weight_tensor(labels, mode, device):
    classes = np.array(sorted(set(labels)))
    balanced = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(labels)).astype(np.float32)
    if mode == 'none':
        return None
    if mode == 'sqrt_balanced':
        applied = np.sqrt(balanced)
    elif mode == 'balanced':
        applied = balanced
    else:
        raise ValueError(f'Unsupported class weight mode: {mode}')
    return torch.tensor(applied, dtype=torch.float, device=device)


class_weights = build_class_weight_tensor(train_labels, config.CLASS_WEIGHT_MODE, config.DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f'Class weights ({config.CLASS_WEIGHT_MODE}): {class_weights.tolist() if class_weights is not None else None}')


def get_training_stage(epoch_number):
    if epoch_number <= config.FREEZE_EPOCHS:
        return 'frozen'
    if epoch_number < config.FULL_UNFREEZE_EPOCH:
        return 'partial'
    return 'full'


def set_phobert_trainable_layers(model, stage):
    for param in model.phobert.parameters():
        param.requires_grad = False
    if stage == 'partial':
        for layer in model.phobert.encoder.layer[-config.PARTIAL_UNFREEZE_LAST_N:]:
            for param in layer.parameters():
                param.requires_grad = True
    elif stage == 'full':
        for param in model.phobert.parameters():
            param.requires_grad = True
    for module in model.get_head_modules():
        for param in module.parameters():
            param.requires_grad = True


def build_optimizer_and_scheduler(model, stage, epoch_number):
    phobert_lr = {'frozen': config.PHOBERT_LR_FROZEN, 'partial': config.PHOBERT_LR_PARTIAL, 'full': config.PHOBERT_LR_FULL}[stage]
    groups = []
    phobert_params = [p for p in model.phobert.parameters() if p.requires_grad]
    if phobert_params:
        groups.append({'params': phobert_params, 'lr': phobert_lr})
    head_params = [p for module in model.get_head_modules() for p in module.parameters() if p.requires_grad]
    groups.append({'params': head_params, 'lr': config.HEAD_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=config.WEIGHT_DECAY)
    remaining_epochs = config.EPOCHS - epoch_number + 1
    total_steps = max(1, remaining_epochs * len(train_loader))
    warmup_steps = int(total_steps * config.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    return optimizer, scheduler, phobert_lr


set_phobert_trainable_layers(model, 'frozen')
print(f'Trainable params after freeze: {count_trainable_params(model):,}')

Class weights (sqrt_balanced): [0.8457201719284058, 2.8837244510650635, 0.8215451836585999]
Trainable params after freeze: 197,635


## 7. Training Functions

In [21]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for batch in tqdm(dataloader, desc='Training', leave=False):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
    }


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    precision_pc, recall_pc, f1_pc, support_pc = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1, 2], zero_division=0)
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1_per_class': f1_pc.tolist(),
        'confusion_matrix': confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]).tolist(),
        'y_pred': list(map(int, all_preds)),
        'y_true': list(map(int, all_labels)),
    }

## 8. Training Loop

In [22]:
history = {'epoch': [], 'stage': [], 'train_loss': [], 'train_acc': [], 'train_f1_macro': [], 'train_f1_weighted': [],
           'val_loss': [], 'val_acc': [], 'val_f1_macro': [], 'val_f1_weighted': [], 'val_f1_neutral': [], 'val_selection_score': []}

best_val_score = -1.0
best_epoch = 0
best_stage = None
best_model_state = None
patience_counter = 0
current_stage = None

print('=' * 70)
print('START TRAINING - PhoBERT Baseline with Positional Encoding')
print('=' * 70)

for epoch in range(1, config.EPOCHS + 1):
    stage = get_training_stage(epoch)
    if stage != current_stage:
        current_stage = stage
        set_phobert_trainable_layers(model, stage)
        optimizer, scheduler, phobert_lr = build_optimizer_and_scheduler(model, stage, epoch)
        print(f'\n[Stage Switch] epoch={epoch} stage={stage} phobert_lr={phobert_lr} trainable_params={count_trainable_params(model):,}')

    print(f'\nEpoch {epoch}/{config.EPOCHS}')
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, scheduler, config.DEVICE)
    val_metrics = evaluate(model, val_loader, criterion, config.DEVICE)
    selection_score = val_metrics[config.SELECTION_METRIC]

    history['epoch'].append(epoch)
    history['stage'].append(stage)
    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['train_f1_macro'].append(train_metrics['f1_macro'])
    history['train_f1_weighted'].append(train_metrics['f1_weighted'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1_macro'].append(val_metrics['f1_macro'])
    history['val_f1_weighted'].append(val_metrics['f1_weighted'])
    history['val_f1_neutral'].append(val_metrics['f1_per_class'][1])
    history['val_selection_score'].append(selection_score)

    print(f"Train  - loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} f1_macro={train_metrics['f1_macro']:.4f}")
    print(f"Val    - loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} f1_macro={val_metrics['f1_macro']:.4f} f1_neutral={val_metrics['f1_per_class'][1]:.4f}")

    if selection_score > best_val_score:
        best_val_score = selection_score
        best_epoch = epoch
        best_stage = stage
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save(best_model_state, os.path.join(config.MODELS_DIR, 'best_model.pt'))
        print(f'  -> New best model saved ({config.SELECTION_METRIC}={best_val_score:.4f})')
    else:
        patience_counter += 1
        print(f'  -> No improvement, patience={patience_counter}/{config.EARLY_STOP_PATIENCE}')
        if patience_counter >= config.EARLY_STOP_PATIENCE:
            print(f'  -> Early stopping at epoch {epoch}')
            break

assert best_model_state is not None
model.load_state_dict(best_model_state)
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'training_history.csv'), index=False)
print(f'\nBest epoch: {best_epoch}, best stage: {best_stage}, best {config.SELECTION_METRIC}: {best_val_score:.4f}')

START TRAINING - PhoBERT Baseline with Positional Encoding

[Stage Switch] epoch=1 stage=frozen phobert_lr=0.0 trainable_params=197,635

Epoch 1/10


KeyboardInterrupt: 

## 9. Evaluation

In [ ]:
val_results = evaluate(model, val_loader, criterion, config.DEVICE)
test_results = evaluate(model, test_loader, criterion, config.DEVICE)

print('Test Classification Report:')
print(classification_report(test_results['y_true'], test_results['y_pred'], target_names=list(config.LABEL_MAP.values()), digits=4))

Test Classification Report:
              precision    recall  f1-score   support

    Negative     0.9318    0.9595    0.9455      1409
     Neutral     0.6345    0.5509    0.5897       167
    Positive     0.9535    0.9415    0.9475      1590

    accuracy                         0.9289      3166
   macro avg     0.8399    0.8173    0.8276      3166
weighted avg     0.9270    0.9289    0.9277      3166



## 10. Save Results

In [ ]:
with open(os.path.join(config.SUMMARIES_DIR, 'training_results.txt'), 'w', encoding='utf-8') as f:
    f.write('=' * 60 + '\n')
    f.write('TRAINING RESULTS - PhoBERT Baseline with Positional Encoding\n')
    f.write('=' * 60 + '\n')
    f.write(f'Best Epoch: {best_epoch}\n')
    f.write(f'Best Stage: {best_stage}\n')
    f.write(f'Selection Metric: {config.SELECTION_METRIC}\n')
    f.write(f'Best Selection Score: {best_val_score:.4f}\n')
    f.write(f'Test Accuracy: {test_results["accuracy"]:.4f}\n')
    f.write(f'Test F1 Macro: {test_results["f1_macro"]:.4f}\n')
    f.write(f'Test F1 Neutral: {test_results["f1_per_class"][1]:.4f}\n')

print(f'Saved outputs to: {config.RESULTS_DIR}')
print('\nFinal Results:')
print(f'Test Accuracy: {test_results["accuracy"]:.4f}')
print(f'Test F1 Macro: {test_results["f1_macro"]:.4f}')
print(f'Test F1 Neutral: {test_results["f1_per_class"][1]:.4f}')

Saved outputs to: /kaggle/working/results/PhoBERT_Baseline_Positional_Fair/improvements/20260321_132228

Final Results:
Test Accuracy: 0.9289
Test F1 Macro: 0.8276
Test F1 Neutral: 0.5897
